In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from pathlib import Path

# 1. Navegación dinámica: Buscamos el archivo pyproject.toml hacia arriba
def find_project_root(current_path, target="pyproject.toml"):
    for parent in Path(current_path).parents:
        if (parent / target).exists():
            return parent
    return None

root = find_project_root(os.getcwd())

if root:
    print(f"📂 Proyecto detectado en: {root}")
    # 2. Instalación en modo editable (-e)
    # El flag --no-deps es opcional si solo quieres registrar los cambios de archivos
    !pip install -e "{root}"
    
    print("\n✅ Instalación completada. Ya puedes importar 'legion_goes' desde cualquier celda.")
else:
    print("❌ Error: No se encontró 'pyproject.toml'. Asegúrate de estar dentro de la estructura de MAIE_tesis_github.")

📂 Proyecto detectado en: /home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github
Obtaining file:///home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for legion-goes (pyproject.toml) ... done
  Created wheel for legion-goes: filename=legion_goes-0.1.9-py3-none-any.whl size=2701 sha256=2cb573997f20247a21bd2324784bd71e814cbb515bee1ecb0bf2c723115accb0
  Stored in directory: /tmp/pip-ephem-wheel-cache-5vmbglyc/wheels/45/f6/84/0a48d0659fd5307d2efb8dfac8c3238f417bd9a968b770cecf
Successfully built legion-goes
  Attempting uninstall: legion-goes
    Found existing installation: legion-goes 0.1.9
    Uninstalling legion-goes-0.1.9:
      Successfully uninstalled legion-goes-0.1.9

✅ Instalación completada. Ya puedes 

In [3]:
try:
    import legion_goes
    print(f"📦 Librería 'legion_goes' lista para usar.")
except ImportError:
    print("⚠️ Instalación terminada, pero puede que necesites reiniciar el Kernel.")

📦 Librería 'legion_goes' lista para usar.


In [5]:
# =============================================================================
# FILE PATH: src/legion_goes/tasks/task02_download/actions/fn_act01/utils.py
# Version: 1.7.0 (Dual Path Logic: Plan vs Raw Data)
# =============================================================================

import json
import itertools
from datetime import datetime, timezone
from pathlib import Path

# SoT Imports
from legion_goes.sot.goes_sat  import get_SOT_sat_info
from legion_goes.sot.goes_prod import get_SOT_product_info

# --- 1. HELPER FUNCTIONS ---

def generate_expected_init_names(product_id: str, sat_id: str, year: str, day: str) -> list:
    """
    Generates a list of expected file start strings (prefixes) based on SoT defaults.
    Example: OR_ABI-L2-LSTF-M6_G16_s20241001200
    """
    # SOT info
    prod_SOT_info = get_SOT_product_info(product_id)
    init_file_name = prod_SOT_info["init_file_name"]
    time_info = prod_SOT_info["default_time"]    
    
    # Basics
    str_year = str(year)
    str_day = str(day).zfill(3)
    date_prefix = f"{str_year}{str_day}"

    # Generate all time combinations (HHMMSS) from SoT
    raw_times = [
        f"{h}{m}{s}".strip() 
        for h, m, s in itertools.product(
            time_info["hours"], 
            time_info["minutes"], 
            time_info["seconds"]
        )
    ]
    
    # Construct prefix: [InitName][SatID]_s
    # Example: OR_ABI-L2-LSTF-M6_G16_s
    file_prefix = f"{init_file_name}{sat_id}_s"
    
    # Combine prefix + date + times
    list_output = [f"{file_prefix}{date_prefix}{t}" for t in raw_times]
    
    return list_output

# ===================================================================
# UNIT TESTING (Main Execution)
# ===================================================================
if __name__ == "__main__":
    print("\n" + " TEST: EXPECTED FILENAME GENERATION ".center(60, "="))
    
    # Mock inputs for testing
    TEST_PRODUCT = "ABI-L2-LSTF"
    TEST_SAT = "19"
    TEST_YEAR = "2026"
    TEST_DAY = "003"

    print(f"▶️  Target Product: {TEST_PRODUCT}")
    print(f"▶️  Target Satellite: GOES-{TEST_SAT}")
    print(f"▶️  Target Date: Year {TEST_YEAR}, Day {TEST_DAY}")

    try:
        # Execute generator
        expected_names = generate_expected_init_names(
            product_id=TEST_PRODUCT,
            sat_id=TEST_SAT,
            year=TEST_YEAR,
            day=TEST_DAY
        )

        # Print results
        total = len(expected_names)
        print(f"\n✅ Successfully generated {total} expected file prefixes.")
        
        print("\n--- Sample of first 5 prefixes ---")
        for name in expected_names[:5]:
            print(f"  - {name}...")

        # Integrity Check: Compare with SoT total_files_one_day
        sot_total = get_SOT_product_info(TEST_PRODUCT)["total_files_one_day"]
        if total == sot_total:
            print(f"\n🛡️  Integrity Check: MATCHES SoT expected count ({sot_total}).")
        else:
            print(f"\n⚠️  Integrity Check: MISMATCH! Generated {total} vs SoT {sot_total}.")

    except Exception as e:
        print(f"\n❌ [CRITICAL ERROR]: {e}")

    print("\n" + "="*60 + "\n")


============ TEST: EXPECTED FILENAME GENERATION ============
▶️  Target Product: ABI-L2-LSTF
▶️  Target Satellite: GOES-19
▶️  Target Date: Year 2026, Day 003

✅ Successfully generated 24 expected file prefixes.

--- Sample of first 5 prefixes ---
  - OR_ABI-L2-LSTF-M6_G19_s202600300...
  - OR_ABI-L2-LSTF-M6_G19_s202600301...
  - OR_ABI-L2-LSTF-M6_G19_s202600302...
  - OR_ABI-L2-LSTF-M6_G19_s202600303...
  - OR_ABI-L2-LSTF-M6_G19_s202600304...

🛡️  Integrity Check: MATCHES SoT expected count (24).




In [ ]:
import os
from pathlib import Path

# =============================================================================
# --- 0. UNIVERSE CONFIGURATION ---
# =============================================================================
# Definimos claramente las dos dimensiones: Control (JSON) y Data (NetCDF)
data_plan_base_path = Path("./data_plan").resolve()
data_raw_base_path  = Path("./data_raw").resolve()

job_params = {
    "sat_id": "19",
    "year": "2026",
    "day": "062",
    "product_id": "ABI-L2-MCMIPF"
}

# =============================================================================
# --- 1. TASK 02: DOWNLOAD (Acquisition) ---
# =============================================================================
from legion_goes.tasks.task02_download.actions.action01_gen_plan_download import run_task02_download_action01_generate_plan
from legion_goes.tasks.task02_download.actions.action02_check_plan_download import execute_task02_download_action02_check_plan
from legion_goes.tasks.task02_download.actions.action03_run_plan_download import execute_task02_download_action03_run_download
from legion_goes.tasks.task02_download.actions.fn01_file_name_plan_download import generate_plan_download_file_path

print("📡 [LEGION] Phase 1: Acquisition Starting...")

# A. GENERATE DOWNLOAD PLAN (Dual Path Mode)
# Vinculamos el JSON al Plan Path y el Inventario al Raw Path
run_task02_download_action01_generate_plan(
    year=job_params["year"],
    day=job_params["day"],
    sat_id=job_params["sat_id"],
    product_id=job_params["product_id"],
    output_folder_base_data_raw=str(data_raw_base_path),   # Donde irán los .nc
    output_folder_base_data_plan=str(data_plan_base_path)  # Donde se guarda el .json
)

# B. LOCATE THE GENERATED PLAN
# Siempre buscamos el archivo físico del plan en la carpeta de Control
path_p_down = generate_plan_download_file_path(
    year=job_params["year"],
    day=job_params["day"],
    sat_id=job_params["sat_id"],
    product_id=job_params["product_id"],
    output_folder_base=str(data_plan_base_path)
)

# C. AUDIT DOWNLOAD PLAN
# El auditor necesita el path de control para leer el JSON
execute_task02_download_action02_check_plan(
    year=job_params["year"],
    day=job_params["day"],
    sat_id=job_params["sat_id"],
    product_id=job_params["product_id"],
    output_folder_base_data_plan=str(data_plan_base_path)
)

# D. EXECUTE DOWNLOAD
# Action 03 simplemente recibe el path del JSON y ejecuta lo que dice dentro
execute_task02_download_action03_run_download(
    path_p_down, 
    threads=10
)

print("\n✅ [DOWNLOAD PHASE COMPLETE]")

In [ ]:
# =============================================================================
# --- 2. TASK 03: PROCESSING (FNP Pipeline) ---
# =============================================================================
from pathlib import Path
from legion_goes.tasks.task03_proc_single.actions.action01_gen_plan_proc_single import run_action01_gen_all_product_plans
from legion_goes.tasks.task03_proc_single.actions.action02_check_plan_proc_single import run_action02_check_plan_proc_single 
from legion_goes.tasks.task03_proc_single.actions.action03_run_plan_proc_single import run_action03_run_plan_proc_single

data_proc_base_path = Path("./data_proc").resolve()

print("\n⚙️ [LEGION] Phase 2: Processing Starting...")

# A. GENERATE PROCESSING PLANS (Action 01)
raw_plans = run_action01_gen_all_product_plans(
    year=job_params["year"], day=job_params["day"],
    sat_id=job_params["sat_id"], product_id=job_params["product_id"],
    output_folder_base_data_raw=str(data_raw_base_path),
    output_folder_base_data_proc=str(data_proc_base_path),
    output_folder_base_data_plan=str(data_plan_base_path)
)

# --- ESTA ES LA PARTE NUEVA QUE DEBES AGREGAR ---
# Filtra la lista para quitar los "None" antes de seguir
list_of_proc_plans = [p for p in raw_plans if p is not None]
# -----------------------------------------------

# B. VALIDATE AND EXECUTE PLANS
if list_of_proc_plans:
    print(f"🚀 Found {len(list_of_proc_plans)} VALID FNP plans to process.")
    
    for path_p_proc in list_of_proc_plans:
        # Ahora path_p_proc NUNCA será None, así que .stem no fallará
        fnp_tag = path_p_proc.stem.split('_')[-1]
        print(f"\n💎 FNP Target: {fnp_tag}")
        
        # --- PASO 1: Action 02 (Audit/Check) ---
        print(f"🔍 [Action 02] Checking plan integrity...")
        is_valid = run_action02_check_plan_proc_single(path_plan_json=str(path_p_proc))
        
        if not is_valid:
            print(f"⚠️ [Action 02] Plan {fnp_tag} has issues. Skipping execution.")
            continue
            
        # --- PASO 2: Action 03 (Run) ---
        print(f"⚡ [Action 03] Starting execution...")
        success = run_action03_run_plan_proc_single(
            path_plan_json=str(path_p_proc), 
            overwrite=False
        )
        
        if success:
            print(f"✅ Plan processed successfully: {path_p_proc.name}")
        else:
            print(f"❌ Error processing plan: {path_p_proc.name}")
else:
    print("⚠️ No processing plans were generated (all returned None).")

print("\n🏆 [LEGION MISSION COMPLETE]")

In [ ]:
cat ./data_plan/2026/062/plan_02-proc-01-single_2026_062_GOES19_EAST_ABI-L2-MCMIPF_fnp01.json | jq '.proc_inventory | to_entries[0]'

In [ ]:
import os
from pathlib import Path

# Sustituye con tus rutas reales del log
raw_path = "/home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github/tests/test_tasks/test_task03_proc_single/test_actions/data_raw"

print(f"¿Existe la carpeta raw?: {os.path.exists(raw_path)}")
if os.path.exists(raw_path):
    archivos = list(Path(raw_path).rglob("*.nc"))
    print(f"Cantidad de archivos .nc encontrados: {len(archivos)}")
    if archivos:
        print(f"Ejemplo del primer archivo: {archivos[0].name}")

In [ ]:
# --- 2. TASK 03: PROCESSING (Fix de Argumentos) ---
from legion_goes.tasks.task03_proc_single.actions.action01_gen_plan_proc_single import run_action01_gen_plan_proc_single
from legion_goes.tasks.task03_proc_single.actions.action02_check_plan_proc_single import run_action02_check_plan_proc_single
from legion_goes.tasks.task03_proc_single.actions.action03_run_plan_proc_single import run_action03_run_plan_proc_single

print("\n⚙️ [FASE 2] Iniciando Procesamiento...")

# A. Generar Plan de Proceso
# Pasamos solo lo que Action01 necesita, evitando el choque con 'output_folder_base'
path_p_proc = run_action01_gen_plan_proc_single(
    year=job_params["year"],
    day=job_params["day"],
    sat_id=job_params["sat_id"],
    product_id=job_params["product_id"],
    fnp_tag="fnp01", 
    path_download_base=str(base_test_path)
)

# B. Check (Auditoría de archivos .nc físicos)
run_action02_check_plan_proc_single(path_p_proc)

# C. Ejecutar el Motor de la Legion (Satpy)
run_action03_run_plan_proc_single(path_p_proc, overwrite=False)

print("\n✅ [PIPELINE COMPLETADO]")

In [13]:
# =============================================================================
# FILE PATH: src/legion_goes/tasks/task01_init/actions/action01_welcome.py
# Version: 1.9.1 (Bugfix: Timezone Import & Precise Dashboard)
# =============================================================================

import os
import sys
import psutil
import shutil
import platform
from datetime import datetime, timezone  


from legion_goes.sot.goes_hardcoded_folders import LEGION_DATA_ROOT

# --- ANSI COLOR PALETTE ---
C_LGN   = "\033[96m"  # Cyan (LEGION)
C_GOS   = "\033[92m"  # Green (GOES)
C_WHT   = "\033[97m"  # White (Bridge)
C_RST   = "\033[0m"   # Reset
C_BLD   = "\033[1m"   # Bold

# =============================================================================
# LEGION VISUAL ASSETS
# =============================================================================

BANNER_01 = r"""
      _      ______ _____ _____ ____  _   _ 
     | |    |  ____/ ____|_   _/ __ \| \ | |
     | |    | |__ | |  __  | || |  | |  \| |
     | |    |  __|| | |_ | | || |  | | . ` |
     | |____| |___| |__| |_| || |__| | |\  |
     |______|______\_____|_____\____/|_| \_|
"""

BANNER_02 = r"""
  _      ______ _____ _____ ____  _   _               _____  ____  ______  _____ 
 | |    |  ____/ ____|_   _/ __ \| \ | |             / ____|/ __ \|  ____|/ ____|
 | |    | |__ | |  __  | || |  | |  \| |  _______   | |  __| |  | | |__  | (___  
 | |    |  __|| | |_ | | || |  | | . ` | |_______|  | | |_ | |  | |  __|  \___ \ 
 | |____| |___| |__| |_| || |__| | |\  |            | |__| | |__| | |____ ____) |
 |______|______\_____|_____\____/|_| \_|             \_____|\____/|______|_____/ 
"""

# =============================================================================
# COLOR ENGINE
# =============================================================================

def get_colored_output(text: str, is_alt: bool = False) -> str:
    if not text: return ""
    lines = text.splitlines()
    colored_lines = []
    split_point = 46 

    for line in lines:
        if not line.strip():
            colored_lines.append("")
            continue
        left = line[:split_point]
        right = line[split_point:]
        if "_______" in right:
            bridge_split = right.split("_______")
            final_line = f"{C_LGN}{left}{C_GOS}{bridge_split[0]}{C_WHT}_______"
            final_line += f"{C_GOS}{bridge_split[1]}{C_RST}"
        else:
            final_line = f"{C_LGN}{left}{C_GOS}{right}{C_RST}"
        colored_lines.append(final_line)
    return "\n".join(colored_lines)

# =============================================================================
# CORE ORCHESTRATOR
# =============================================================================

def show_welcome_banner(use_alt: bool = False):
    """Displays the colored welcome banner and detailed system diagnostics."""
    # 1. Tiempos
    now_local = datetime.now()
    now_utc = datetime.now(timezone.utc)
    
    # 2. RAM (Libre of Total | % Free)
    ram = psutil.virtual_memory()
    ram_total_gb = ram.total / (1024**3)
    ram_avail_gb = ram.available / (1024**3)
    ram_free_pct = (ram.available / ram.total) * 100
    
    # 3. DISCO (Libre of Total | % Free)
    total, used, free = shutil.disk_usage(LEGION_DATA_ROOT)
    disk_total_gb = total / (1024**3)
    disk_free_gb = free / (1024**3)
    disk_free_pct = (free / total) * 100

    raw_art = BANNER_01 if use_alt else BANNER_02
    
    print("\n" + "="*95)
    print(get_colored_output(raw_art, is_alt=use_alt))
    print("="*95)
    
    # --- DASHBOARD TÉCNICO ---
    print(f"  {C_BLD}SYSTEM DATE:{C_RST}           {now_local.strftime('%Y-%m-%d %H:%M:%S')} (Local)")
    print(f"  {C_BLD}UTC DATE:{C_RST}              {now_utc.strftime('%Y-%m-%d %H:%M:%S')} (Z)")
    print(f"  {C_BLD}OPERATING SYSTEM:{C_RST}      {platform.system()} {platform.release()} ({platform.machine()})")
    print(f"  {C_BLD}WORKSPACE:{C_RST}             {LEGION_DATA_ROOT}")
    
    # RAM: 10.00 GB of 32.00 GB (31.2% free)
    print(f"  {C_BLD}SYSTEM RAM:{C_RST}            {ram_avail_gb:.2f} GB of {ram_total_gb:.2f} GB ({ram_free_pct:.1f}% free)")
    
    # STORAGE: 100.00 GB of 1000.00 GB (10.0% free)
    print(f"  {C_BLD}SYSTEM STORAGE:{C_RST}        {disk_free_gb:.2f} GB of {disk_total_gb:.2f} GB ({disk_free_pct:.1f}% free)")
    
    print("="*95 + "\n")

if __name__ == "__main__":
    show_welcome_banner()



  _      ______ _____ _____ ____  _   _               _____  ____  ______  _____ 
 | |    |  ____/ ____|_   _/ __ \| \ | |             / ____|/ __ \|  ____|/ ____|
 | |    | |__ | |  __  | || |  | |  \| |  _______   | |  __| |  | | |__  | (___  
 | |    |  __|| | |_ | | || |  | | . ` | |_______|  | | |_ | |  | |  __|  \___ \ 
 | |____| |___| |__| |_| || |__| | |\  |            | |__| | |__| | |____ ____) |
 |______|______\_____|_____\____/|_| \_|             \_____|\____/|______|_____/ 
  SYSTEM DATE:           2026-03-06 20:33:34 (Local)
  UTC DATE:              2026-03-06 19:33:34 (Z)
  OPERATING SYSTEM:      Linux 6.17.0-14-generic (x86_64)
  WORKSPACE:             /home/legion/bulk/MAIE_tesis2026/f01_code/MAIE_tesis_github/tests/test_tasks/test_task03_proc_single/test_actions
  SYSTEM RAM:            21.28 GB of 30.78 GB (69.2% free)
  SYSTEM STORAGE:        290.56 GB of 591.38 GB (49.1% free)



In [14]:
# =============================================================================
# FILE PATH: src/legion_goes/tasks/task01_init/actions/action02_folders.py
# Version: 1.0.0 (Folder Management Action)
# =============================================================================

import os
from legion_goes.sot.goes_hardcoded_folders import LEGION_DATA_ROOT, GOES_FOLDERS

def create_folder_structure():
    """Verifica y crea la estructura de directorios necesaria."""
    
    print(f"[SYSTEM] Checking minimal folder structure environment...")
    
    for folder_key, folder_path in GOES_FOLDERS.items():
        # --- AQUÍ FILTRAMOS LA QUE NO QUIERES ---
        # Sustituye "NOMBRE_A_EXCLUIR" por el nombre exacto de la key en el SoT
        if folder_key == "NOMBRE_A_EXCLUIR":
            continue 
            
        full_path = os.path.join(LEGION_DATA_ROOT, folder_path)
        
        # Crear si no existe
        os.makedirs(full_path, exist_ok=True)
        
        print(f"  + Checking directory: {folder_key} -> OK")
    
    print(f"[SUCCESS] LEGION-GOES Environment ready for processing.\n")

if __name__ == "__main__":
    create_folder_structure()

[SYSTEM] Checking minimal folder structure environment...
  + Checking directory: root -> OK
  + Checking directory: data_raw -> OK
  + Checking directory: data_plan -> OK
  + Checking directory: data_proc -> OK
  + Checking directory: reports -> OK
  + Checking directory: logs -> OK
[SUCCESS] LEGION-GOES Environment ready for processing.

